In [6]:
import numpy as np, matplotlib.pyplot as plt

In [7]:
import numpy as np

# ground truth boxes
gt_boxes = np.array([
    [50, 50, 200, 200],
    [220, 220, 370, 370],
    [400, 50, 500, 150]
])

# predicted boxes
pred_boxes = np.array([
    [48, 48, 198, 198],
    [60, 60, 210, 210],
    [215, 215, 365, 365],
    [390, 40, 510, 160]
])

# confidence
scores = np.array([0.9, 0.75, 0.8, 0.6])

iou_threshold = 0.5

In [8]:
# sort predictions by confidence
indices = np.argsort(scores)[::-1]
pred_boxes = pred_boxes[indices]
scores = scores[indices]

TP = np.zeros(len(pred_boxes))
FP = np.zeros(len(pred_boxes))
matched_gt = []

In [9]:
# compute TP/FP
for i, pred in enumerate(pred_boxes):
    ious = []
    for gt in gt_boxes:
        # iou calculation
        x1 = max(pred[0], gt[0])
        y1 = max(pred[1], gt[1])
        x2 = min(pred[2], gt[2])
        y2 = min(pred[3], gt[3])
        if x2 < x1 or y2 < y1:
            iou = 0
        else:
            inter = (x2-x1)*(y2-y1)
            union = (pred[2]-pred[0])*(pred[3]-pred[1]) + (gt[2]-gt[0])*(gt[3]-gt[1]) - inter
            iou = inter/union
        ious.append(iou)
    max_iou = max(ious)
    max_idx = np.argmax(ious)
    if max_iou >= iou_threshold and max_idx not in matched_gt:
        TP[i] = 1
        matched_gt.append(max_idx)
    else:
        FP[i] = 1

In [24]:
# compute precision and recall
cum_TP = np.cumsum(TP)
cum_FP = np.cumsum(FP)
precision = cum_TP / (cum_TP + cum_FP)
recall = cum_TP / len(gt_boxes)

print("Precision:", precision)
print("Recall:", recall)

Precision: [1.         1.         0.66666667 0.75      ]
Recall: [0.33333333 0.66666667 0.66666667 1.        ]


In [26]:
# compute Average Precision - AP
AP = 0
for i in range(len(precision)):
    if i == 0:
        delta_recall = recall[i]
    else:
        delta_recall = recall[i] - recall[i-1]
    AP += precision[i] * delta_recall

print(f"average precision - AP: {AP:.4f}")

# mAP for multiple classes
# APs for 3 dummy classes
APs = [AP, 0.85, 0.90]
mAP = np.mean(APs)
print(f"mean average precision - mAP: {mAP:.4f}")

average precision - AP: 0.9167
mean average precision - mAP: 0.8889
